In [4]:
from tqdm.notebook import tqdm
import numpy as np
import torch
from matplotlib import animation, pyplot as plt

import math
from inspect import isfunction
from functools import partial

from tqdm.auto import tqdm
from einops import rearrange

import torch
from torch import nn, einsum
import torch.nn.functional as F
from imagegen.setup import find_root, load_config
import imagegen.unet as unet
import imagegen.train as train
import imagegen.data as data
import imagegen.ddpmeval as ddpmeval

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
X = torch.rand(size=(2, 4, 16, 16))

In [6]:
X.size()

torch.Size([2, 4, 16, 16])

In [7]:

class Attention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=32):
        super().__init__()
        self.scale = dim_head**-0.5
        self.heads = heads
        hidden_dim = dim_head * heads
        self.to_qkv = nn.Conv2d(dim, hidden_dim * 3, 1, bias=False)
        self.to_out = nn.Conv2d(hidden_dim, dim, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.to_qkv(x).chunk(3, dim=1)
        q, k, v = map(
            lambda t: rearrange(t, "b (h c) x y -> b h c (x y)", h=self.heads), qkv
        )
        q = q * self.scale

        sim = einsum("b h d i, b h d j -> b h i j", q, k)
        sim = sim - sim.amax(dim=-1, keepdim=True).detach()  # cool numerical instability trick - nograd on this
        attn = sim.softmax(dim=-1)

        out = einsum("b h i j, b h d j -> b h i d", attn, v)
        out = rearrange(out, "b h (x y) d -> b (h d) x y", x=h, y=w)
        return self.to_out(out)


In [35]:
x = X; dim_head=32
b, c, h, w = x.shape
qkv = attn.to_qkv(x).chunk(3, dim=1)
q, k, v = map(
    lambda t: rearrange(t, "b (h c) x y -> b h c (x y)", h=4), qkv
)
q = q * (dim_head**-0.5)

sim = einsum("b h d i, b h d j -> b h i j", q, k)


In [36]:
qkv[0].size()

torch.Size([2, 128, 16, 16])

In [37]:
q.size() # 2 4 32 256

torch.Size([2, 4, 32, 256])

In [30]:
rearrange(X, "b (h c) x y -> b h c (x y)", h=1).size()

torch.Size([2, 1, 4, 256])

In [11]:
attn = Attention(dim=4)

In [12]:
attn(X).size()

torch.Size([2, 4, 16, 16])

In [15]:
attn.to_qkv

Conv2d(4, 384, kernel_size=(1, 1), stride=(1, 1), bias=False)

In [17]:
attn.to_out

Conv2d(128, 4, kernel_size=(1, 1), stride=(1, 1))

In [20]:
qkv = attn.to_qkv(X).chunk(3, dim=1)

In [21]:
q,k,v = qkv

In [22]:
q.size()

torch.Size([2, 128, 16, 16])

In [23]:
k.size()

torch.Size([2, 128, 16, 16])

In [24]:

v.size()

torch.Size([2, 128, 16, 16])